# Synthesis and Capture

We'll now examine how to generate and capture signals with the RF hardware integrated into the RFSoC. 

In [1]:
from acadia.system import Acadia
from acadia.channel import Channel

We'll start by attaching to the board:

In [2]:
acadia = Acadia()
acadia.attach()

Next, we'll set up the clocking system:

In [3]:
acadia.configure_clocks(reference="internal")

We'll make sure that everything started up okay:

In [ ]:
acadia.get_clock_status()

In [ ]:
Channel.RFDC_status()

In [4]:
acadia = Acadia()

pulse_time = 100e-9

pulse_channel = acadia.DAC(0)
pulse_length = pulse_channel.seconds_to_samples(pulse_time)
pulse_memory = acadia.DACArray[pulse_channel.num](size=pulse_channel.seconds_to_bytes(pulse_time))

# Create a sequence for the sequencer
@acadia.sequence
def sequence(a):
    with a.sequencer() as seq:
        with seq.loop():
            with a.synchronizer(block=False):
                a.generate(pulse_channel, pulse_memory)

            with seq.wait_until(a.channels_almost_done(pulse_channel)):
                pass

# Instruct the PS to create a 100ns cosine pulse and load it into memory
def program():   
    import numpy as np
    import time
    
    # Load the pulses into DAC memory
    pulse = np.ones(pulse_length, dtype=np.complex64)
    pulse_samples = pulse_channel.to_samples(pulse)
    acadia.memcpy(pulse_samples, pulse_memory)
    
    # Set up the channel properties
    pulse_channel.set_nyquist_zone(2)
    pulse_channel.configure_nco(frequency=1205e6)
    pulse_channel.set_vop(4000)

    # Configure the ADC switch
    acadia.configure()

    # Reset and run the sequencer
    acadia.sequencer_reset()
    acadia.sequencer_run(sequence)
    time.sleep(0.1)
    

In [5]:
acadia.compile_all()

In [6]:
acadia.attach()
acadia.assemble(load=True)

In [7]:
program()

NameError: name 'capture_channel' is not defined

In [ ]:
acadia.sequencer_halt()
    